# Teaching a Sales Pipeline to Learn: PPO from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/pipeline-ppo/blob/main/notebooks/pipeline_ppo_codealong.ipynb)

**What you'll build:** a reinforcement learning agent that learns which sales play to run on each open deal, every week, to maximize net revenue. It starts as a copy of a sensible human playbook and learns past it.

**Why it matters:** this is the same algorithm used to train chat models with human feedback (RLHF). Every piece maps one to one:

| RLHF for language models | This project |
|---|---|
| Prompt plus the tokens written so far | A deal: stage, fit, engagement, weeks open, size |
| Next token | Next play: nurture email, SDR call, case study, exec briefing, discount, disqualify |
| Supervised fine-tuned (SFT) model | A policy cloned from the current sales playbook |
| Reward model score | Closed revenue minus the cost of the plays |
| Value head V(s) | **Opportunity value**: expected net revenue from this point |
| KL penalty to the SFT model | KL penalty to the playbook clone |
| PPO clipped update | Same |

**What you'll learn**
1. How a value function turns into a live opportunity forecast, and how the value loss trains it.
2. How advantages, generalized advantage estimation (GAE) and PPO's clipped objective decide which plays to reinforce.
3. Why RLHF keeps the model close to a reference policy, and what happens when you loosen that leash.

**Time:** 45 to 60 minutes. Training takes about two minutes on Colab's free CPU. No GPU, no deep learning framework: the neural network and its backpropagation are written in plain NumPy so nothing is hidden.

> The market is simulated, with rules the agent never sees. That's what lets us check whether it learned the right lessons. Part 10 covers what changes with real CRM data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field

PAPER, INK, RED, RIVER, BRICK, CONCRETE, GRAPHITE = "#F3EEE4", "#121212", "#D02418", "#1F3A4D", "#8E3B26", "#8D8A83", "#4A4741"
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
np.set_printoptions(precision=3, suppress=True)
GAMMA = 0.99   # weekly discount used when scoring forecasts
print("Ready")

## Part 1. The simulated pipeline

Each **episode is one opportunity**. Every week the agent picks one play for each open deal. Three things can happen that week: the deal advances a stage, it's lost, or nothing changes. A deal ends when it closes, is lost, is disqualified or stalls at 26 weeks.

**What the agent can observe (the state):** stage, ICP fit, buyer engagement, weeks open, deal size, whether a discount was offered, whether an executive sponsor is in place, and last week's play.

**What the agent can't observe (the hidden rules):**
- SDR calls work early; case studies work mid-funnel.
- Repeating the same play two weeks running works half as well (buyer fatigue).
- An exec briefing only pays off on **strong-fit, late-stage** deals, where it wins an executive sponsor who expands the deal 25%. On weak-fit deals it wastes $3K.
- A discount only rescues **cold** negotiations (engagement below 0.5), and always gives up 15% of revenue.
- Every open week costs $150 of rep attention.

The current playbook ignores those nuances: exec briefing at every Proposal, discount at every Negotiation. The question is whether PPO can find the nuance from outcomes alone.

**Reward** is closed revenue minus play costs, in units of $100K. Like RLHF, it's sparse: almost all of it arrives at the end of the episode.

In [ ]:
STAGES = ["Discovery", "Qualified", "Solution", "Proposal", "Negotiation"]
ACTIONS = ["Nurture email", "SDR call", "Case study", "Exec briefing", "Discount 15%", "Disqualify"]
N_STAGES, N_ACTIONS = len(STAGES), len(ACTIONS)

# Cost of each play in $K of team time and money.
ACTION_COST = np.array([0.05, 0.4, 0.6, 3.0, 0.0, 0.0])
HOLDING_COST = 0.15        # $K per open deal per week (rep attention, forecasting overhead)
DISCOUNT = 0.15            # revenue given up when a discount is offered
SPONSOR_UPLIFT = 0.25      # scope expansion when an executive sponsor is secured
MAX_WEEKS = 26             # deals that haven't closed in two quarters are counted as stalled

# Engagement lift from each play at each stage (rows = actions, cols = stages).
# These are the hidden "physics" of the market. The agent never sees this table.
ENGAGE = np.array([
    # Disc  Qual  Sol   Prop  Neg
    [0.03, 0.03, 0.02, 0.01, 0.01],   # Nurture email
    [0.22, 0.18, 0.06, 0.03, 0.02],   # SDR call
    [0.05, 0.14, 0.24, 0.10, 0.04],   # Case study
    [0.02, 0.04, 0.08, 0.10, 0.10],   # Exec briefing (big effect comes through ADVANCE, for high-fit deals)
    [0.00, 0.00, 0.02, 0.04, 0.06],   # Discount
    [0.00, 0.00, 0.00, 0.00, 0.00],   # Disqualify
])
# Direct push on the chance of advancing a stage (logit units).
ADVANCE = np.array([
    [0.00, 0.00, 0.00, 0.00, 0.00],
    [0.35, 0.30, 0.00, 0.00, 0.00],
    [0.00, 0.20, 0.45, 0.10, 0.00],
    [0.00, 0.00, 0.10, 0.20, 0.20],   # plus a fit-dependent bonus below
    [0.00, 0.00, 0.00, 0.30, 0.70],   # plus a low-engagement bonus at Negotiation
    [0.00, 0.00, 0.00, 0.00, 0.00],
])

LAST0 = 11                       # index where the last-play one-hot starts
OBS_DIM = LAST0 + N_ACTIONS

In [ ]:
class PipelineEnv:
    """Vectorized pipeline simulator.

    Observation per deal (what a rep or CRM would know):
      stage one-hot (5), fit, engagement, weeks open / 26, log deal size (scaled),
      discount already given (0/1), executive sponsor secured (0/1), last play one-hot (6).
    Reward: closed revenue minus the cost of plays and holding cost, in $K / 100.
    """

    def __init__(self, n_deals: int = 256, seed: int = 0, reward_scale: float = 100.0):
        self.n = n_deals
        self.rng = np.random.default_rng(seed)
        self.scale = reward_scale
        self.stage = np.zeros(n_deals, dtype=int)
        self.fit = np.zeros(n_deals)
        self.eng = np.zeros(n_deals)
        self.week = np.zeros(n_deals, dtype=int)
        self.size = np.zeros(n_deals)
        self.disc = np.zeros(n_deals)
        self.sponsor = np.zeros(n_deals)
        self.last = np.zeros(n_deals, dtype=int)
        self._new(np.arange(n_deals))

    # ---- deal generation -------------------------------------------------
    def _new(self, idx: np.ndarray) -> None:
        k = len(idx)
        if k == 0:
            return
        self.stage[idx] = 0
        self.fit[idx] = self.rng.beta(2.0, 2.0, k)                     # ICP fit score, 0..1
        self.eng[idx] = self.rng.uniform(0.05, 0.35, k)                # starting engagement
        self.week[idx] = 0
        self.size[idx] = np.exp(self.rng.normal(np.log(55), 0.45, k))  # deal size in $K, median ~55K
        self.disc[idx] = 0.0
        self.sponsor[idx] = 0.0
        self.last[idx] = 0

    def reset(self) -> np.ndarray:
        self._new(np.arange(self.n))
        return self.obs()

    def obs(self) -> np.ndarray:
        o = np.zeros((self.n, OBS_DIM))
        o[np.arange(self.n), self.stage] = 1.0
        o[:, 5] = self.fit
        o[:, 6] = self.eng
        o[:, 7] = self.week / MAX_WEEKS
        o[:, 8] = (np.log(self.size) - np.log(55)) / 0.45
        o[:, 9] = self.disc
        o[:, 10] = self.sponsor
        o[np.arange(self.n), LAST0 + self.last] = 1.0
        return o

    # ---- one week ---------------------------------------------------------
    def step(self, a: np.ndarray):
        n, r = self.n, self.rng
        s = self.stage.copy()
        reward = -ACTION_COST[a] - HOLDING_COST
        done = np.zeros(n, dtype=bool)
        won = np.zeros(n, dtype=bool)

        # Disqualify ends the deal immediately with no revenue.
        dq = a == 5
        done |= dq

        # Engagement: repeated plays fatigue the buyer, so the same play twice in a row works half as well.
        fatigue = np.where(a == self.last, 0.5, 1.0)
        lift = ENGAGE[a, s] * fatigue * (0.6 + 0.8 * self.fit)
        self.eng = np.clip(self.eng + lift * (1 - self.eng) - 0.04 * self.eng, 0, 1)

        # A discount anywhere sets the flag; the revenue cut applies at close.
        self.disc = np.where(a == 4, 1.0, self.disc)

        # Chance of advancing a stage this week.
        logit = -2.6 + 2.2 * self.fit + 2.6 * self.eng + ADVANCE[a, s] * fatigue
        exec_bonus = (a == 3) * (s >= 3) * np.where(self.fit > 0.6, 0.6, -0.3)   # executives only help strong-fit late deals
        disc_bonus = (a == 4) * (s == 4) * np.where(self.eng < 0.5, 0.9, 0.0)     # discounts only rescue cold negotiations
        # An exec briefing on a strong-fit, late-stage deal wins an executive sponsor, who expands the scope of the deal.
        self.sponsor = np.where((a == 3) & (s >= 3) & (self.fit > 0.6), 1.0, self.sponsor)
        logit = logit + exec_bonus + disc_bonus
        p_adv = 1 / (1 + np.exp(-logit))
        p_lost = 1 / (1 + np.exp(-(-4.2 + 2.2 * (1 - self.eng) + 1.8 * (1 - self.fit) + 0.06 * self.week)))

        u = r.random(n)
        lost = (u < p_lost) & ~dq
        adv = (~lost) & (r.random(n) < p_adv) & ~dq

        close = adv & (s == 4)
        won |= close
        value = self.size * (1 + SPONSOR_UPLIFT * self.sponsor) * (1 - DISCOUNT * self.disc)
        reward = reward + np.where(close, value, 0.0)
        self.stage = np.where(adv & (s < 4), s + 1, s)

        done |= lost | close
        self.week += 1
        stalled = (self.week >= MAX_WEEKS) & ~done
        done |= stalled
        self.last = a.copy()

        info = {"won": won, "lost": lost, "disqualified": dq, "stalled": stalled,
                "revenue": np.where(close, value, 0.0), "sponsor": self.sponsor.copy(),
                "cost": ACTION_COST[a] + HOLDING_COST, "weeks": self.week.copy(), "fit": self.fit.copy()}
        # Replace finished deals with new ones so every slot keeps producing experience.
        self._new(np.where(done)[0])
        return self.obs(), reward / self.scale, done, info

In [ ]:
def playbook_action(obs: np.ndarray) -> np.ndarray:
    """The 'current playbook': a sensible human rule set, one play per stage.

    Discovery/Qualified: SDR call. Solution: case study. Proposal: exec briefing.
    Negotiation: discount. Never disqualify. Alternates with a nurture email when
    the same play was just used, to avoid obvious fatigue.
    """
    stage = obs[:, :N_STAGES].argmax(1)
    last = obs[:, LAST0:].argmax(1)
    play = np.array([1, 1, 2, 3, 4])[stage]
    return np.where(play == last, 0, play)

Watch one deal move through the pipeline under the current playbook.

In [ ]:
env = PipelineEnv(n_deals=1, seed=4)
obs = env.reset()
print(f"New deal: fit {env.fit[0]:.2f}, engagement {env.eng[0]:.2f}, size ${env.size[0]:.0f}K")
for week in range(26):
    a = playbook_action(obs)
    stage = STAGES[env.stage[0]]
    obs, r, done, info = env.step(a)
    eng = f"engagement {obs[0,6]:.2f}" if not done[0] else ""
    print(f"week {week+1:2d}  {stage:<12} play: {ACTIONS[a[0]]:<14} reward ${r[0]*100:7.2f}K  {eng}")
    if done[0]:
        print("Outcome:", "WON" if info["won"][0] else "lost" if info["lost"][0] else "disqualified/stalled")
        break

**Check your understanding**
1. Why is the reward negative almost every week?
2. Where does this setup look like RLHF's "one score at the end of the completion"?

<details><summary>Answers</summary>

1. Every play has a cost and every open week has a $150 holding cost. Revenue only arrives on the week a deal closes.
2. Most steps earn nothing (or a small cost), and the big signal comes only when the episode ends. That makes credit assignment hard: which of the eight plays earned the win? Value functions and GAE exist to solve exactly this.
</details>

## Part 2. Baselines: how good is the playbook?

Before training anything, measure the bar we need to beat. `evaluate` runs a policy on thousands of fresh deals and keeps score per deal.

In [ ]:
def evaluate(policy, n_deals: int = 4000, seed: int = 123, parallel: int = 500):
    """policy(obs) -> actions. Returns per-deal outcomes as a dict of arrays."""
    env = PipelineEnv(parallel, seed=seed)
    obs = env.reset()
    cost = np.zeros(parallel)
    used = np.zeros((parallel, N_ACTIONS))
    start_fit = env.fit.copy()
    out = {k: [] for k in ["revenue", "cost", "won", "weeks", "disqualified", "discounted", "exec_briefings", "fit", "net"]}
    while len(out["net"]) < n_deals:
        a = policy(obs)
        disc_before = np.maximum(env.disc, (a == 4).astype(float))
        obs, _, d, info = env.step(a)
        cost += info["cost"]
        used[np.arange(parallel), a] += 1
        for i in np.where(d)[0]:
            out["revenue"].append(info["revenue"][i]); out["cost"].append(cost[i]); out["won"].append(info["won"][i])
            out["weeks"].append(info["weeks"][i]); out["disqualified"].append(info["disqualified"][i])
            out["discounted"].append(bool(disc_before[i]) and info["won"][i]); out["exec_briefings"].append(used[i, 3])
            out["fit"].append(start_fit[i]); out["net"].append(info["revenue"][i] - cost[i])
            cost[i] = 0; used[i] = 0; start_fit[i] = env.fit[i]
    return {k: np.array(v[:n_deals]) for k, v in out.items()}

In [ ]:
def summarize(res: dict) -> dict:
    won = res["won"]
    return {
        "net_per_deal": res["net"].mean(),
        "win_rate": won.mean(),
        "revenue_per_deal": res["revenue"].mean(),
        "cost_per_deal": res["cost"].mean(),
        "weeks_to_close": res["weeks"][won].mean() if won.any() else np.nan,
        "discount_rate_on_wins": res["discounted"][won].mean() if won.any() else np.nan,
        "exec_briefings_per_deal": res["exec_briefings"].mean(),
        "disqualify_rate": res["disqualified"].mean(),
    }

In [ ]:
policies = {
    "Current playbook": playbook_action,
    "Random plays": lambda o: np.random.default_rng(0).integers(0, 5, len(o)),
    "Nurture only": lambda o: np.zeros(len(o), dtype=int),
}
baseline = pd.DataFrame({name: summarize(evaluate(p, 6000, seed=2026)) for name, p in policies.items()}).T
baseline.round(2)

The playbook beats random and lazy policies by a wide margin, so it isn't a straw man. Note two things: it discounts **every** win and runs an exec briefing on roughly half of all deals. Keep those numbers in mind.

## Part 3. A neural network in plain NumPy

The agent needs two networks, just like an RLHF setup:
- an **actor** (the policy) that outputs a score (logit) for each play, turned into probabilities with softmax;
- a **critic** (the value head) that outputs one number, V(s), the expected net revenue from this state.

Both are small two-layer networks. `backward` is the chain rule written out by hand: given how the loss changes with the output, it returns how the loss changes with every weight. `adam` then nudges the weights.

In [ ]:
class MLP:
    def __init__(self, n_in: int, n_out: int, hidden: int = 64, seed: int = 0, out_scale: float = 0.01):
        rng = np.random.default_rng(seed)
        def init(i, o, s=1.0):
            return rng.normal(0, s * np.sqrt(1.0 / i), (i, o))
        self.p = {"W1": init(n_in, hidden), "b1": np.zeros(hidden),
                  "W2": init(hidden, hidden), "b2": np.zeros(hidden),
                  "W3": init(hidden, n_out, out_scale), "b3": np.zeros(n_out)}
        self.m = {k: np.zeros_like(v) for k, v in self.p.items()}
        self.v = {k: np.zeros_like(v) for k, v in self.p.items()}
        self.t = 0

    def forward(self, x: np.ndarray):
        p = self.p
        h1 = np.tanh(x @ p["W1"] + p["b1"])
        h2 = np.tanh(h1 @ p["W2"] + p["b2"])
        out = h2 @ p["W3"] + p["b3"]
        return out, (x, h1, h2)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.forward(x)[0]

    def backward(self, cache, d_out: np.ndarray) -> dict:
        """d_out = dLoss/dOutput (already averaged over the batch). Returns parameter gradients."""
        x, h1, h2 = cache
        p = self.p
        g = {"W3": h2.T @ d_out, "b3": d_out.sum(0)}
        d_h2 = (d_out @ p["W3"].T) * (1 - h2 ** 2)
        g["W2"], g["b2"] = h1.T @ d_h2, d_h2.sum(0)
        d_h1 = (d_h2 @ p["W2"].T) * (1 - h1 ** 2)
        g["W1"], g["b1"] = x.T @ d_h1, d_h1.sum(0)
        return g

    def adam(self, g: dict, lr: float = 3e-4, b1: float = 0.9, b2: float = 0.999, eps: float = 1e-8, clip: float = 0.5):
        # Global gradient-norm clipping, as in most PPO implementations.
        norm = np.sqrt(sum((v ** 2).sum() for v in g.values()))
        if norm > clip:
            g = {k: v * clip / norm for k, v in g.items()}
        self.t += 1
        for k in self.p:
            self.m[k] = b1 * self.m[k] + (1 - b1) * g[k]
            self.v[k] = b2 * self.v[k] + (1 - b2) * g[k] ** 2
            mh = self.m[k] / (1 - b1 ** self.t)
            vh = self.v[k] / (1 - b2 ** self.t)
            self.p[k] -= lr * mh / (np.sqrt(vh) + eps)

    def copy(self) -> "MLP":
        c = MLP.__new__(MLP)
        c.p = {k: v.copy() for k, v in self.p.items()}
        c.m = {k: np.zeros_like(v) for k, v in self.p.items()}
        c.v = {k: np.zeros_like(v) for k, v in self.p.items()}
        c.t = 0
        return c

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def log_softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))

**Always gradient-check hand-written backprop.** Nudge one weight, measure the change in output, and compare with what `backward` claims.

In [ ]:
m = MLP(5, 3, hidden=8, seed=0, out_scale=1.0)
x = np.random.default_rng(1).normal(size=(4, 5))
out, cache = m.forward(x)
grads = m.backward(cache, np.ones_like(out))          # gradient of sum(outputs)
for key, idx in [("W1", (1, 2)), ("W2", (3, 4)), ("b3", (1,))]:
    eps = 1e-6
    m.p[key][idx] += eps; up = m(x).sum()
    m.p[key][idx] -= 2 * eps; down = m(x).sum()
    m.p[key][idx] += eps
    print(f"{key}{idx}: backprop {grads[key][idx]:+.6f}   numerical {(up - down) / (2 * eps):+.6f}")

## Part 4. Step one of RLHF: clone the playbook

RLHF never starts PPO from a random model. It starts from a supervised fine-tuned (SFT) model that already behaves sensibly, then keeps a frozen copy of it as the **reference policy**.

We do the same: train the actor to imitate the playbook with cross-entropy. Label smoothing keeps a sliver of probability on other plays so PPO still has room to explore.

In [ ]:
def behavior_clone(actor: MLP, steps: int = 300, n_deals: int = 512, seed: int = 1, lr: float = 1e-3, smoothing: float = 0.2):
    """Train the actor to imitate the playbook with cross-entropy, like SFT before RLHF.

    Label smoothing keeps a little probability on other plays, so PPO still has room to explore.
    """
    env = PipelineEnv(n_deals, seed=seed)
    obs = env.reset()
    for _ in range(steps):
        a = playbook_action(obs)
        logits, cache = actor.forward(obs)
        target = np.full_like(logits, smoothing / N_ACTIONS)
        target[np.arange(len(a)), a] += 1.0 - smoothing
        d = softmax(logits) - target                # dCE/dlogits = pi - target
        actor.adam(actor.backward(cache, d / len(a)), lr=lr, clip=1.0)
        obs, _, _, _ = env.step(a)
    acc = (actor(obs).argmax(1) == playbook_action(obs)).mean()
    return float(acc)

In [ ]:
actor_demo = MLP(OBS_DIM, N_ACTIONS, 64, seed=0, out_scale=0.01)
acc = behavior_clone(actor_demo)
print(f"Cloned policy matches the playbook on {acc:.0%} of states")

## Part 5. The value function and the value loss

This is the equation from the PPO lesson:

$$L^{VF} = \frac{1}{2}\left\| V_\theta(s) - \left(\sum_{t=0}^{T} \gamma^t r_t \;\middle|\; s_0 = s\right)\right\|_2^2$$

- $V_\theta(s)$ is the critic's prediction for a deal in state $s$.
- The sum is the discounted net revenue that actually followed.
- $\gamma$ = 0.99 per week, so revenue eight weeks out counts about 92 cents on the dollar.

**Worked example (units of $100K).** A Stage 2 deal: the critic predicts **0.34** ($34K). The deal closes eight weeks later for $187K after $5K of plays. Discounted return ≈ 0.99⁸ × 1.87 − 0.05 ≈ 1.68.

In [ ]:
V_pred, ret = 0.34, 0.99**8 * 1.87 - 0.05
print(f"return = {ret:.3f},  value loss = 0.5 * ({V_pred} - {ret:.3f})^2 = {0.5 * (V_pred - ret)**2:.3f}")
print(f"advantage = return - V = {ret - V_pred:+.3f}  -> this deal did far better than forecast")

In practice PPO doesn't wait for every deal to finish. It uses **generalized advantage estimation (GAE)**, which blends real rewards with the critic's own estimate of later weeks:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t), \qquad A_t = \delta_t + \gamma\lambda A_{t+1}$$

- $\delta_t$ is the one-week surprise: what happened this week plus what we now expect, minus what we expected before.
- $\lambda$ (0.95) controls how far ahead to trust real rewards versus the critic. $\lambda = 1$ is the full sum on the slide (unbiased, noisy); $\lambda = 0$ trusts the critic after one step (stable, biased).
- The value target is $A_t + V(s_t)$, and the critic is trained with exactly the loss above.

In [ ]:
def gae(rew, val, done, last_val, gamma: float, lam: float):
    """Generalized advantage estimation.

    delta_t = r_t + gamma * V(s_{t+1}) * (1 - done_t) - V(s_t)
    A_t     = delta_t + gamma * lam * (1 - done_t) * A_{t+1}
    The value target ("return") is A_t + V(s_t).
    """
    T = len(rew)
    adv = np.zeros_like(rew)
    nxt_adv = np.zeros_like(last_val)
    nxt_val = last_val
    for t in reversed(range(T)):
        nonterm = 1.0 - done[t]
        delta = rew[t] + gamma * nxt_val * nonterm - val[t]
        nxt_adv = delta + gamma * lam * nonterm * nxt_adv
        adv[t] = nxt_adv
        nxt_val = val[t]
    return adv, adv + val

In [ ]:
# Tiny check: a 4-week deal, reward only at the end, gamma = lambda = 1 and a critic that predicts 0
rew  = np.array([[0.0], [0.0], [0.0], [1.87]])
val  = np.zeros((4, 1)); done = np.array([[0.], [0.], [0.], [1.]])
adv, ret = gae(rew, val, done, last_val=np.zeros(1), gamma=1.0, lam=1.0)
print("returns each week:", ret.ravel(), " (every week gets credit for the final 1.87)")
adv, ret = gae(rew, val, done, last_val=np.zeros(1), gamma=0.99, lam=0.95)
print("with gamma=0.99, lambda=0.95:", ret.ravel().round(3))

**Check your understanding**
1. If the critic's predictions were perfect, what would the average advantage be?
2. Why does a better critic make the policy learn faster?

<details><summary>Answers</summary>

1. Zero. Advantage measures surprise against the forecast, and a perfect forecast is never surprised on average.
2. Advantages become less noisy. The policy update is "do more of what beat the forecast"; a bad forecast mislabels lucky deals as good decisions and unlucky ones as bad decisions.
</details>

## Part 6. The PPO update

For each play the agent took, compare how likely that play is **now** versus when the experience was collected:

$$r(\theta) = \frac{\pi_\theta(a\mid s)}{\pi_{old}(a\mid s)}$$

PPO maximizes $\min\big(r A,\ \text{clip}(r, 1-\epsilon, 1+\epsilon) A\big)$. If a play had a positive advantage, its probability can rise, but only by about 20% per update. If negative, it can fall, also by about 20%. The clip is what keeps a lucky batch of deals from rewriting the whole playbook.

Two more terms, both straight from RLHF:
- **Entropy bonus**: keeps some randomness so the agent keeps exploring.
- **KL penalty to the reference** (the playbook clone): every move away from the playbook has a price, $\beta \cdot KL(\pi \,\|\, \pi_{ref})$.

In [ ]:
r = np.linspace(0.5, 1.5, 200)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, A, title in [(axes[0], 1.0, "Positive advantage (A = +1)"), (axes[1], -1.0, "Negative advantage (A = -1)")]:
    ax.plot(r, r * A, color=CONCRETE, ls="--", label="unclipped r·A")
    ax.plot(r, np.minimum(r * A, np.clip(r, 0.8, 1.2) * A), color=RED, lw=2, label="PPO objective")
    ax.axvline(1, color=INK, lw=0.5); ax.set_title(title, loc="left", fontweight="bold"); ax.set_xlabel("probability ratio r")
axes[0].legend(frameon=False); plt.tight_layout(); plt.show()

Flat sections of the red line have zero gradient: once a play has moved 20%, this batch can't push it further. Here is the full update, with every gradient written out:

In [ ]:
@dataclass
class PPOConfig:
    n_deals: int = 256          # deals simulated in parallel
    horizon: int = 32           # weeks per rollout
    iterations: int = 120
    epochs: int = 4
    minibatch: int = 2048
    gamma: float = 0.99         # weekly discount
    lam: float = 0.95           # GAE lambda
    clip: float = 0.2           # PPO clip range (epsilon)
    vf_coef: float = 1.0
    ent_coef: float = 0.01
    kl_coef: float = 0.02       # beta: pull toward the playbook clone (the "reference model")
    lr_actor: float = 3e-4
    lr_critic: float = 1e-3
    hidden: int = 64
    seed: int = 0

In [ ]:
def rollout(env: PipelineEnv, obs: np.ndarray, actor: MLP, critic: MLP, horizon: int, rng: np.random.Generator):
    n = env.n
    buf = {k: [] for k in ["obs", "act", "logp", "val", "rew", "done"]}
    done_rev, done_cost = [], []
    for _ in range(horizon):
        logits = actor(obs)
        logp_all = log_softmax(logits)
        p = np.exp(logp_all)
        a = (p.cumsum(1) > rng.random((n, 1))).argmax(1)      # sample one action per deal
        v = critic(obs)[:, 0]
        nxt, r, d, info = env.step(a)
        buf["obs"].append(obs); buf["act"].append(a); buf["logp"].append(logp_all[np.arange(n), a])
        buf["val"].append(v); buf["rew"].append(r); buf["done"].append(d.astype(float))
        obs = nxt
    last_val = critic(obs)[:, 0]
    return {k: np.array(v) for k, v in buf.items()}, last_val, obs

In [ ]:
def ppo_update(actor: MLP, critic: MLP, ref: MLP, batch: dict, cfg: PPOConfig, rng: np.random.Generator):
    N = len(batch["act"])
    stats = {"clip_frac": [], "approx_kl": [], "entropy": [], "kl_ref": [], "value_loss": []}
    for _ in range(cfg.epochs):
        idx = rng.permutation(N)
        for s in range(0, N, cfg.minibatch):
            j = idx[s:s + cfg.minibatch]
            o, a, old_logp = batch["obs"][j], batch["act"][j], batch["logp"][j]
            A = batch["adv"][j]
            A = (A - A.mean()) / (A.std() + 1e-8)
            R = batch["ret"][j]
            B = len(j)

            # ---- policy (actor) ----
            logits, cache = actor.forward(o)
            logp_all = log_softmax(logits)
            pi = np.exp(logp_all)
            logp = logp_all[np.arange(B), a]
            ratio = np.exp(logp - old_logp)
            clipped = np.clip(ratio, 1 - cfg.clip, 1 + cfg.clip)
            # Objective = min(ratio*A, clip(ratio)*A). Gradient flows only where the unclipped term is the min.
            use_unclipped = (ratio * A) <= (clipped * A)
            g_logp = np.where(use_unclipped, ratio * A, 0.0)     # d objective / d logp(a)
            onehot = np.zeros_like(pi); onehot[np.arange(B), a] = 1.0
            d_obj = g_logp[:, None] * (onehot - pi)              # d objective / d logits

            # Entropy bonus: H = -sum pi log pi ; dH/dz_j = -pi_j (log pi_j + H)
            H = -(pi * logp_all).sum(1)
            d_H = -pi * (logp_all + H[:, None])

            # KL to the reference (playbook clone): KL = sum pi (log pi - log pi_ref)
            ref_logp = log_softmax(ref(o))
            kl = (pi * (logp_all - ref_logp)).sum(1)
            d_kl = pi * ((logp_all - ref_logp) - kl[:, None])

            # Minimize: -objective - ent_coef*H + kl_coef*KL
            d_logits = (-d_obj - cfg.ent_coef * d_H + cfg.kl_coef * d_kl) / B
            actor.adam(actor.backward(cache, d_logits), lr=cfg.lr_actor)

            # ---- value (critic): L_VF = 1/2 (V(s) - R)^2, the loss on the slide ----
            v, vcache = critic.forward(o)
            err = v[:, 0] - R
            critic.adam(critic.backward(vcache, (cfg.vf_coef * err / B)[:, None]), lr=cfg.lr_critic)

            stats["clip_frac"].append(float((~use_unclipped).mean()))
            stats["approx_kl"].append(float((old_logp - logp).mean()))
            stats["entropy"].append(float(H.mean()))
            stats["kl_ref"].append(float(kl.mean()))
            stats["value_loss"].append(float(0.5 * (err ** 2).mean()))
    return {k: float(np.mean(v)) for k, v in stats.items()}

## Part 7. Train the agent

`train` runs the whole RLHF-style loop: clone the playbook, freeze a reference copy, then alternate between collecting 32 weeks of experience on 256 deals and running the PPO update. It prints a fresh evaluation every 25 iterations. Expect about two minutes.

In [ ]:
def greedy(actor: MLP):
    return lambda o: actor(o).argmax(1)


def train(cfg: PPOConfig = PPOConfig(), verbose: bool = True, eval_every: int = 10):
    rng = np.random.default_rng(cfg.seed)
    actor = MLP(OBS_DIM, N_ACTIONS, cfg.hidden, seed=cfg.seed, out_scale=0.01)
    critic = MLP(OBS_DIM, 1, cfg.hidden, seed=cfg.seed + 1, out_scale=1.0)
    bc_acc = behavior_clone(actor, seed=cfg.seed + 7)
    ref = actor.copy()                                   # frozen reference policy, like the SFT model in RLHF
    env = PipelineEnv(cfg.n_deals, seed=cfg.seed + 11)
    obs = env.reset()
    hist = []
    for it in range(1, cfg.iterations + 1):
        buf, last_val, obs = rollout(env, obs, actor, critic, cfg.horizon, rng)
        adv, ret = gae(buf["rew"], buf["val"], buf["done"], last_val, cfg.gamma, cfg.lam)
        batch = {"obs": buf["obs"].reshape(-1, OBS_DIM), "act": buf["act"].reshape(-1),
                 "logp": buf["logp"].reshape(-1), "adv": adv.reshape(-1), "ret": ret.reshape(-1)}
        ev = 1 - np.var(ret - buf["val"]) / (np.var(ret) + 1e-8)
        stats = ppo_update(actor, critic, ref, batch, cfg, rng)
        row = {"iteration": it, "explained_variance": float(ev), **stats}
        if it % eval_every == 0 or it == 1:
            row["eval_net_per_deal"] = float(evaluate(greedy(actor), n_deals=3000, seed=999)["net"].mean())
            if verbose:
                print(f"iter {it:4d}  net/deal ${row['eval_net_per_deal']:6.2f}K  value loss {stats['value_loss']:.4f}  "
                      f"KL to playbook {stats['kl_ref']:.3f}  clip {stats['clip_frac']:.2f}  EV {ev:.2f}")
        hist.append(row)
    return actor, critic, ref, hist, bc_acc

In [ ]:
cfg = PPOConfig(iterations=300, lr_actor=1e-3)
actor, critic, ref, hist, bc_acc = train(cfg, eval_every=25)
hist = pd.DataFrame(hist)

**Reading the training log**
- **net/deal** should climb above the playbook within the first 25 to 50 iterations.
- **value loss** is the equation from Part 5, averaged over the batch.
- **KL to playbook** shows how far the agent has moved from the reference. It should rise and then level off: the penalty stops it drifting forever.
- **clip** is the share of samples where the clip was active. Single-digit percentages mean small, steady updates.
- **EV** (explained variance) is how much of the variation in returns the critic explains. 1.0 is perfect; 0 means no better than the average. Deals are noisy, so 0.5 is respectable.

## Part 8. Did it beat the playbook, and what did it learn?

In [ ]:
res_pb = evaluate(playbook_action, 8000, seed=2026)
res_ppo = evaluate(greedy(actor), 8000, seed=2026)
comp = pd.DataFrame({"Current playbook": summarize(res_pb), "PPO agent": summarize(res_ppo)})
lift = comp.loc["net_per_deal", "PPO agent"] / comp.loc["net_per_deal", "Current playbook"] - 1
print(f"Net value per deal: {lift:+.1%}")
comp.round(3)

In [ ]:
def action_mix_by_stage(policy, n_steps: int = 60, seed: int = 5):
    """Share of each play chosen at each stage, over the states the policy actually visits."""
    env = PipelineEnv(1000, seed=seed)
    obs = env.reset()
    counts = np.zeros((len(STAGES), N_ACTIONS))
    for _ in range(n_steps):
        a = policy(obs)
        st = obs[:, :5].argmax(1)
        np.add.at(counts, (st, a), 1)
        obs, *_ = env.step(a)
    return counts / counts.sum(1, keepdims=True)


def probe_state(stage: int, fit: float, eng: float, week: float = 0.3, last: int = 0):
    o = np.zeros(OBS_DIM)
    o[stage] = 1; o[5] = fit; o[6] = eng; o[7] = week; o[LAST0 + last] = 1
    return o

In [ ]:
mix_pb, mix_ppo = action_mix_by_stage(playbook_action), action_mix_by_stage(greedy(actor))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, mix, title in [(axes[0], mix_pb, "Current playbook"), (axes[1], mix_ppo, "PPO agent")]:
    ax.imshow(mix, cmap="Reds", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(N_ACTIONS)); ax.set_xticklabels(ACTIONS, rotation=30, ha="right")
    ax.set_yticks(range(N_STAGES)); ax.set_yticklabels(STAGES); ax.set_title(title, loc="left", fontweight="bold")
    for i in range(N_STAGES):
        for j in range(N_ACTIONS):
            if mix[i, j] >= 0.05:
                ax.text(j, i, f"{mix[i, j]:.0%}", ha="center", va="center", fontsize=8, color=PAPER if mix[i, j] > 0.55 else INK)
plt.tight_layout(); plt.show()

Did it discover the hidden rules? Probe the policy directly: hold everything fixed and vary one thing.

In [ ]:
fits, engs = np.linspace(0, 1, 41), np.linspace(0, 1, 41)
p_exec = np.exp(log_softmax(actor(np.array([probe_state(3, f, 0.5) for f in fits]))))[:, 3]
p_disc = np.exp(log_softmax(actor(np.array([probe_state(4, 0.45, e) for e in engs]))))[:, 4]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(fits, p_exec, color=RED, lw=2); axes[0].axvline(0.6, color=CONCRETE, ls=":")
axes[0].set_title("P(exec briefing) at Proposal vs ICP fit", loc="left", fontweight="bold"); axes[0].set_ylim(0, 1)
axes[1].plot(engs, p_disc, color=RIVER, lw=2); axes[1].axvline(0.5, color=CONCRETE, ls=":")
axes[1].set_title("P(discount) at Negotiation vs engagement", loc="left", fontweight="bold"); axes[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

The dotted lines are the simulator's hidden thresholds. The agent was never told them; it found them from outcomes. It learned to **target** exec briefings at strong-fit deals and to **save** discounts for cold negotiations.

**Check your understanding**
1. The agent's win rate is about the same as the playbook's, yet net value is clearly higher. How?
2. The early-stage rows of the heatmap barely changed. Why might PPO leave them alone?

<details><summary>Answers</summary>

1. It stopped giving away 15% on deals that would have closed anyway, and it targeted exec sponsors at deals where the 25% expansion was available. Win rate is the wrong goal; net value is the right one.
2. Two reasons. The playbook was already close to right there, so the advantages are small. And the KL penalty charges for every departure from the reference, so the agent only deviates where it pays.
</details>

## Part 9. The value head is an opportunity forecast

Here is the payoff for a marketing or RevOps team, even if you never let an agent pick plays. The critic, trained only with the value loss, is a **dynamic opportunity value**: expected net revenue from this exact deal state, updated every week.

Compare it with the classic stage-weighted forecast ("every Stage 2 deal is worth the Stage 2 average").

In [ ]:
def snapshot_values(actor, critic, n_steps: int = 220, parallel: int = 1500, seed: int = 77):
    """Run the stochastic policy, record V(s) at every weekly snapshot and the discounted net revenue that followed."""
    env = PipelineEnv(parallel, seed=seed)
    obs = env.reset()
    rng = np.random.default_rng(seed)
    rec_v, rec_stage, rec_r, rec_d, rec_slot = [], [], [], [], []
    for _ in range(n_steps):
        p = np.exp(log_softmax(actor(obs)))
        a = (p.cumsum(1) > rng.random((parallel, 1))).argmax(1)
        rec_v.append(critic(obs)[:, 0] * env.scale); rec_stage.append(obs[:, :5].argmax(1))
        obs, r, d, _ = env.step(a)
        rec_r.append(r * env.scale); rec_d.append(d)
    V, S, R, D = map(np.array, (rec_v, rec_stage, rec_r, rec_d))
    T = len(R)
    G = np.zeros_like(R); complete = np.zeros_like(D, dtype=bool)
    g = np.zeros(parallel); ended = np.zeros(parallel, dtype=bool)
    for t in reversed(range(T)):
        g = R[t] + GAMMA * g * (1 - D[t])
        ended = D[t] | ended                                  # the deal seen at week t finishes at or after t
        G[t] = g; complete[t] = ended
    keep = complete                                           # only snapshots whose deal finished inside the window
    return V[keep], S[keep], G[keep]

In [ ]:
V, S, G = snapshot_values(actor, critic)
static = pd.Series(G).groupby(S).mean().reindex(S).to_numpy()
r2 = lambda pred: 1 - ((G - pred) ** 2).mean() / G.var()
print(f"R² of the value head:     {r2(V):.2f}")
print(f"R² of stage averages:     {r2(static):.2f}")
bins = pd.qcut(V, 10, duplicates="drop")
cal = pd.DataFrame({"pred": V, "actual": G}).groupby(bins, observed=True).mean()
fig, ax = plt.subplots(figsize=(5, 4.5))
lim = [cal.min().min() - 2, cal.max().max() + 2]
ax.plot(lim, lim, color=CONCRETE, ls="--"); ax.scatter(cal["pred"], cal["actual"], color=RED, s=50, zorder=3)
ax.set_xlabel("Predicted V(s), $K"); ax.set_ylabel("Actual discounted net, $K")
ax.set_title("Calibration by decile", loc="left", fontweight="bold"); plt.tight_layout(); plt.show()

Points on the diagonal mean the forecast is calibrated: deals the critic values at $30K return about $30K on average. Individual deals are still uncertain (an R² near 0.4 at the deal level is strong for a single opportunity), but in aggregate it's a forecast you can plan against.

## Part 10. Experiments

Each run below uses 120 iterations to stay quick. Try them, then design your own.

**Experiment A: loosen or tighten the leash.** RLHF's KL coefficient trades reward against staying close to the reference. What happens to net value and KL as β changes?

In [ ]:
rows = []
for beta in [0.0, 0.02, 0.2]:
    a_, c_, r_, h_, _ = train(PPOConfig(iterations=120, lr_actor=1e-3, kl_coef=beta), verbose=False, eval_every=120)
    s_ = summarize(evaluate(greedy(a_), 4000, seed=2026))
    rows.append({"kl_coef": beta, "net_per_deal": s_["net_per_deal"], "discount_rate_on_wins": s_["discount_rate_on_wins"],
                 "final_KL_to_playbook": h_[-1]["kl_ref"]})
pd.DataFrame(rows).round(3)

**Experiment B (try it yourself): change the market.** Set `ACTION_COST[3] = 6.0` (exec briefings get more expensive), rerun Part 7 with fewer iterations, and probe the exec-briefing curve again. Does the threshold move right?

**Experiment C: why does the agent never disqualify?** Disqualifying saves holding costs on hopeless deals. Try raising `HOLDING_COST` to 0.6 and see whether disqualification appears for low-fit deals.

**Experiment D: remove the reference.** Train with `kl_coef=0.0` and `ent_coef=0.0`. Watch the action mix collapse toward a few plays. This is the language-model equivalent of reward hacking and mode collapse.

## What transfers to real pipelines

- **The value head works on historical data today.** Train V(s) on weekly CRM snapshots with the value loss, and you get dynamic opportunity value without ever letting an agent act.
- **PPO itself needs a simulator or careful experiments.** It learns by trying. On live deals, start with contextual bandits and randomized holdouts, and use historical data to build the simulator.
- **Logged CRM data is biased.** Reps already send exec briefings to the deals they expect to win, so naive analysis overstates the play. That's the same causal problem as incrementality testing.
- **The KL leash is a governance feature.** It's how you let an AI change a sales motion without letting it rewrite it overnight.

Built by **Na'im McKee**. Companion projects: geo incrementality testing, marketing mix modeling and a self-auditing ad budget agent.